# Summer Analytics 2026 - Week 2 Hackathon
## E-Commerce Conversion Prediction Challenge

**Goal**: Predict whether a user converts (binary classification)  
**Metric**: F1 Score  
**Approach**: 
1. Combine `train` + `public_test` for maximum training data
2. Feature engineering (Engagement Score interaction)
3. **Ensemble**: Weighted average of simple XGBoost and balanced Random Forest
4. Threshold optimization to maximize F1 score

### Step 1: Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

### Step 2: Load Data

In [2]:
train = pd.read_csv("train.csv")
public_test = pd.read_csv("public_test.csv")
private_test_raw = pd.read_csv("private_test.csv")

print("Train Shape:", train.shape)
print("Public Test Shape:", public_test.shape)
print("Private Test Shape:", private_test_raw.shape)

# Combine train and public_test for more training data
full_train = pd.concat([train, public_test], ignore_index=True)
print("\nCombined Training Data:", full_train.shape)
print("Conversion Rate:", round(full_train['Converted'].mean(), 4))

Train Shape: (10000, 14)
Public Test Shape: (3000, 14)
Private Test Shape: (3000, 13)

Combined Training Data: (13000, 14)
Conversion Rate: 0.3056


### Step 3: Exploratory Data Analysis

In [3]:
print("=== Missing Values ===")
print(full_train.isnull().sum())
print("\n=== Target Distribution ===")
print(full_train['Converted'].value_counts())
print("\n=== Data Types ===")
print(full_train.dtypes)
print("\n=== Categorical Unique Values ===")
print("Device_Type:", full_train['Device_Type'].unique())
print("Traffic_Source:", full_train['Traffic_Source'].unique())

=== Missing Values ===
User_ID                  0
Age                   1952
Income                1285
City_Tier                0
Device_Type              0
Traffic_Source           0
Pages_Viewed             0
Products_Viewed          0
Time_On_Site          2416
Previous_Purchases       0
Discount_Seen            0
Browser_Version          0
Campaign_Code            0
Converted                0
dtype: int64

=== Target Distribution ===
Converted
0    9027
1    3973
Name: count, dtype: int64

=== Data Types ===
User_ID                 int64
Age                   float64
Income                float64
City_Tier               int64
Device_Type            object
Traffic_Source         object
Pages_Viewed            int64
Products_Viewed         int64
Time_On_Site          float64
Previous_Purchases      int64
Discount_Seen           int64
Browser_Version         int64
Campaign_Code           int64
Converted               int64
dtype: object

=== Categorical Unique Values ===
Device_Type:

### Step 4: Feature Engineering

In [4]:
def engineer_features(df):
    """Create new features from existing columns."""
    data = df.copy()
    
    # Core interaction feature
    data['Engagement_Score'] = data['Pages_Viewed'] * data['Products_Viewed']
    
    return data

full_train = engineer_features(full_train)
private_test = engineer_features(private_test_raw)

print("Features after engineering:", len(full_train.columns))
print("New features:", [c for c in full_train.columns if c not in train.columns])

Features after engineering: 15
New features: ['Engagement_Score']


### Step 5: Preprocessing

In [5]:
TARGET = "Converted"

# Encode categorical columns
cat_cols = ['Device_Type', 'Traffic_Source']
for col in cat_cols:
    le = LabelEncoder()
    all_vals = pd.concat([full_train[col], private_test[col]]).astype(str)
    le.fit(all_vals)
    full_train[col] = le.transform(full_train[col].astype(str))
    private_test[col] = le.transform(private_test[col].astype(str))
    print(f"{col} classes: {le.classes_}")

# Separate features and target
y = full_train[TARGET]
X = full_train.drop(columns=["User_ID", TARGET])
X_test = private_test.drop(columns=["User_ID"])

# Fill missing values with median
for col in X.columns:
    med = X[col].median()
    X[col] = X[col].fillna(med)
    X_test[col] = X_test[col].fillna(med)

print(f"\nTraining: {X.shape}, Test: {X_test.shape}")
print(f"Missing: train={X.isnull().sum().sum()}, test={X_test.isnull().sum().sum()}")

Device_Type classes: ['Desktop' 'Mobile' 'Tablet']
Traffic_Source classes: ['Email' 'Organic' 'Paid Ads' 'Referral' 'Social Media']

Training: (13000, 13), Test: (3000, 13)
Missing: train=0, test=0


### Step 6: Define Models (XGBoost + Random Forest Ensemble)

We use a weighted ensemble of a simple **XGBoost** and a balanced **Random Forest**.

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Calculate class imbalance for XGBoost
scale_pos = (y == 0).sum() / (y == 1).sum()

# 1. XGBoost (Simple)
xgb = XGBClassifier(
    n_estimators=100, 
    max_depth=5, 
    learning_rate=0.1,
    scale_pos_weight=scale_pos,
    random_state=42, 
    eval_metric='logloss'
)

# 2. Random Forest (Balanced)
rf = RandomForestClassifier(
    n_estimators=100, 
    class_weight='balanced', 
    random_state=42, 
    n_jobs=-1
)

### Step 7: Cross-Validation & Threshold Optimization

We use cross-validation to get out-of-fold probability predictions for both models. Then we calculate a **weighted average** (XGBoost has 2x weight because it is slightly stronger). Finally, we find the exact probability threshold that maximizes the F1 Score.

In [7]:
print("Generating cross-validation probabilities...")
probs_xgb = cross_val_predict(xgb, X, y, cv=cv, method='predict_proba')[:, 1]
probs_rf = cross_val_predict(rf, X, y, cv=cv, method='predict_proba')[:, 1]

# Weighted average (2 parts XGBoost, 1 part Random Forest)
ensemble_probs = (2 * probs_xgb + probs_rf) / 3

print("Optimizing threshold for maximum F1...")
best_t = 0.5
best_f1 = 0

for t in np.arange(0.20, 0.65, 0.005):
    preds = (ensemble_probs >= t).astype(int)
    score = f1_score(y, preds)
    if score > best_f1:
        best_f1 = score
        best_t = t

print(f"\nBest Threshold: {best_t:.3f}")
print(f"Best CV F1 Score: {best_f1:.4f}")

cv_preds = (ensemble_probs >= best_t).astype(int)
print("\nClassification Report (Cross-Validated):")
print(classification_report(y, cv_preds))

Generating cross-validation probabilities...


Optimizing threshold for maximum F1...



Best Threshold: 0.380
Best CV F1 Score: 0.5599

Classification Report (Cross-Validated):
              precision    recall  f1-score   support

           0       0.85      0.58      0.69      9027
           1       0.44      0.76      0.56      3973

    accuracy                           0.63     13000
   macro avg       0.64      0.67      0.62     13000
weighted avg       0.72      0.63      0.65     13000



### Step 8: Train Final Models & Predict

In [8]:
# Train models on all available data
xgb.fit(X, y)
rf.fit(X, y)
print("Models trained on all data!")

# Generate test probabilities
test_probs_xgb = xgb.predict_proba(X_test)[:, 1]
test_probs_rf = rf.predict_proba(X_test)[:, 1]

# Apply weights and threshold
final_test_probs = (2 * test_probs_xgb + test_probs_rf) / 3
final_predictions = (final_test_probs >= best_t).astype(int)

print(f"\nFinal Predictions (threshold={best_t:.3f}):")
print(f"  Not Converted (0): {(final_predictions == 0).sum()}")
print(f"  Converted (1): {(final_predictions == 1).sum()}")
print(f"  Predicted conversion rate: {final_predictions.mean():.4f}")

Models trained on all data!

Final Predictions (threshold=0.380):
  Not Converted (0): 1403
  Converted (1): 1597
  Predicted conversion rate: 0.5323


### Step 9: Create Submission File

In [10]:
submission = pd.DataFrame({
    "User_ID": private_test_raw["User_ID"],
    "Converted": final_predictions
})

submission.to_csv("submission.csv", index=False)
print("submission.csv created successfully!")
print(f"Shape: {submission.shape}")
submission.head(10)

submission.csv created successfully!
Shape: (3000, 2)


,User_ID,Converted
0,103001,0
1,103002,0
2,103003,1
3,103004,1
4,103005,0
5,103006,0
6,103007,1
7,103008,0
8,103009,0
9,103010,0
